In [ ]:
import scanpy as sc
import anndata as ad
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import InfoGlobe
from InfoGlobe.metrics import fisher_rao_dis_matrix
from InfoGlobe.utils import get_knn


In [ ]:
adata = sc.read_h5ad('sim_data/adata/adata_1.h5ad')

In [ ]:
adata

In [ ]:
P = adata.X
P = P / P.sum(axis=1, keepdims=True)
P_gd = P.T

In [ ]:
sns.heatmap(P, cmap="coolwarm")

In [ ]:
n, m = P_gd.shape
k = 3

In [ ]:
gb = InfoGlobe.infoglobe.GlobeEmbedding(A = [n,k], Q = [k,m], c=1)

In [ ]:
gb.fit(torch.Tensor(P_gd), max_iter=20000)

In [ ]:
x = [i*10 for i in range(len(gb.loss1))]               

plt.plot(x, gb.loss1, label='Reconstruction Loss')               
plt.plot(x, gb.loss2, label='MDS Loss')                   

plt.xlabel("Iteration")
plt.ylabel("Loss")         
plt.legend()
plt.show()      

In [ ]:
adata.obsm['infoglobe_embedding'] = gb.Q.detach().cpu().numpy().T
adata.varm['infoglobe_kernel'] = gb.A.detach().cpu().numpy()

In [ ]:
sns.heatmap(adata.obsm['infoglobe_embedding'][:,[1,0,2]], cmap="coolwarm")

In [ ]:
fisher_dis_mat = fisher_rao_dis_matrix(torch.tensor(adata.obsm['infoglobe_embedding'].T))
dis_mat, knn_mat = get_knn(fisher_dis_mat, k=50)

adata.obsp['connectivities'] = knn_mat
adata.obsp['distances'] = dis_mat

In [ ]:
adata.uns['neighbors'] = {
    'params': {
        'n_neighbors': 50,
        'method': 'umap',
        'metric': 'fisher_rao',
        # 可以添加其他 Scanpy 期望的参数，如：
        'use_rep': 'infoglobe_embedding', 
    },
    
    # 保持不变：指向 obsp 中的矩阵
    'connectivities_key': 'connectivities',
    'distances_key': 'distances',
}
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata,)